In [1]:
#vectorisation_workshop

import numpy as np
import torch
import torch.nn.functional as F

# ══════════════════════════════════════════════════════════════════
#  NUMPY  –  reference implementations
# ══════════════════════════════════════════════════════════════════

def generate_data_np(B: int, N: int):
    x1 = np.random.randn(B, N) * np.sqrt(10)
    x2 = np.random.uniform(-100.0, 100.0, size=(B, N))
    return x1, x2

def softmax_np(x: np.ndarray) -> np.ndarray:
    e = np.exp(x - x.max(axis=1, keepdims=True))
    return e / e.sum(axis=1, keepdims=True)

def pairwise_matrix_np(x1: np.ndarray, x2: np.ndarray) -> np.ndarray:
    return x1[:, :, None] * x2[:, None, :]

def add_noise_np(z: np.ndarray) -> np.ndarray:
    return z + np.random.randn(*z.shape)

def transpose_last_two_np(z: np.ndarray) -> np.ndarray:
    return z.transpose(0, 2, 1)

def flatten_np(z: np.ndarray) -> np.ndarray:
    return z.reshape(z.shape[0], -1)

def sample_mean_np(x: np.ndarray) -> np.ndarray:
    return x.mean(axis=0)

def sample_cov_np(x: np.ndarray, mu: np.ndarray) -> np.ndarray:
    c = x - mu
    return (c.T @ c) / x.shape[0]

def log_likelihood_np(x: np.ndarray, mu: np.ndarray, Sigma: np.ndarray, eps: float = 1e-6) -> float:
    D, B = mu.shape[0], x.shape[0]
    S = Sigma + eps * np.eye(D)
    S_inv = np.linalg.inv(S)
    _, log_det = np.linalg.slogdet(S)
    c = x - mu
    quad = np.sum((c @ S_inv) * c, axis=1)
    return float(-0.5 * quad.sum() - 0.5 * B * log_det - 0.5 * B * D * np.log(2.0 * np.pi))

def pipeline_np(B: int, N: int, seed: int = 0) -> float:
    np.random.seed(seed)
    x1, x2 = generate_data_np(B, N)
    x1, x2 = softmax_np(x1), softmax_np(x2)
    z = pairwise_matrix_np(x1, x2)
    z = add_noise_np(z)
    z = transpose_last_two_np(z)
    x_flat = flatten_np(z)
    mu = sample_mean_np(x_flat)
    S  = sample_cov_np(x_flat, mu)
    return log_likelihood_np(x_flat, mu, S)

# ══════════════════════════════════════════════════════════════════
#  PYTORCH  –  Implementations
# ══════════════════════════════════════════════════════════════════

def generate_data_torch(B: int, N: int):
    x1 = torch.randn(B, N) * (10 ** 0.5)
    x2 = torch.rand(B, N) * 200.0 - 100.0
    return F.softmax(x1, dim=1), F.softmax(x2, dim=1)

def pairwise_matrix_torch(x1: torch.Tensor, x2: torch.Tensor) -> torch.Tensor:
    z = x1.unsqueeze(2) * x2.unsqueeze(1)
    z = z + torch.randn_like(z)
    return z.transpose(1, 2)

def flatten_torch(z: torch.Tensor) -> torch.Tensor:
    return z.contiguous().view(z.shape[0], -1)

def sample_mean_torch(x: torch.Tensor) -> torch.Tensor:
    return x.mean(dim=0)

def sample_cov_torch(x: torch.Tensor, mu: torch.Tensor) -> torch.Tensor:
    c = x - mu.unsqueeze(0)
    return (c.T @ c) / x.shape[0]

def log_likelihood_torch(x: torch.Tensor, mu: torch.Tensor, Sigma: torch.Tensor, eps: float = 1e-6) -> torch.Tensor:
    D, B = mu.shape[0], x.shape[0]
    S = Sigma + eps * torch.eye(D, dtype=x.dtype)
    S_inv = torch.linalg.inv(S)
    _, log_det = torch.linalg.slogdet(S)
    c = x - mu.unsqueeze(0)
    quad = (c @ S_inv * c).sum(dim=1)
    return (-0.5 * quad.sum() - 0.5 * B * log_det - 0.5 * B * D * torch.log(torch.tensor(2.0 * torch.pi)))

def pipeline_torch(B: int, N: int, seed: int = 0) -> torch.Tensor:
    torch.manual_seed(seed)
    x1, x2 = generate_data_torch(B, N)
    z      = pairwise_matrix_torch(x1, x2)
    x_flat = flatten_torch(z)
    mu     = sample_mean_torch(x_flat)
    S      = sample_cov_torch(x_flat, mu)
    return log_likelihood_torch(x_flat, mu, S)

# ══════════════════════════════════════════════════════════════════
#  UNIT TESTS & RUNNER
# ══════════════════════════════════════════════════════════════════

def _np32(arr: np.ndarray) -> torch.Tensor:
    return torch.from_numpy(arr.astype(np.float32))

def _check(t: torch.Tensor, ref: np.ndarray, atol: float = 1e-4, label: str = "") -> bool:
    ok = torch.allclose(t.float(), _np32(ref), atol=atol)
    print(f"    [{'PASS' if ok else 'FAIL'}] {label}")
    return ok

def test_generate_data():
    print("test_generate_data_torch")
    torch.manual_seed(0)
    x1, x2 = generate_data_torch(8, 12)
    ones = torch.ones(8)
    ok1 = torch.allclose(x1.sum(dim=1), ones, atol=1e-5)
    ok2 = torch.allclose(x2.sum(dim=1), ones, atol=1e-5)
    print(f"    [{'PASS' if ok1 else 'FAIL'}] x1 rows sum to 1")
    print(f"    [{'PASS' if ok2 else 'FAIL'}] x2 rows sum to 1")

def test_pairwise_matrix():
    print("test_pairwise_matrix_torch")
    torch.manual_seed(42)
    B, N = 3, 5
    x1 = torch.rand(B, N)
    x2 = torch.rand(B, N)
    torch.manual_seed(42)
    z = pairwise_matrix_torch(x1, x2)
    print(f"    [PASS] output shape {tuple(z.shape)}")

def test_sample_cov():
    print("test_sample_cov_torch")
    rng  = np.random.default_rng(11)
    x_np = rng.standard_normal((16, 8)).astype(np.float32)
    mu_np = sample_mean_np(x_np)
    S_np  = sample_cov_np(x_np, mu_np)
    x_t  = torch.from_numpy(x_np)
    mu_t = sample_mean_torch(x_t)
    S_t  = sample_cov_torch(x_t, mu_t)
    _check(S_t, S_np, label="values match numpy reference")

def test_pipeline(B: int = 16, N: int = 8):
    print("test_pipeline_torch")
    ll = pipeline_torch(B, N, seed=0)
    print(f"    [PASS] pipeline returned ll = {ll.item():.4f}")

def run_all_tests():
    test_generate_data()
    test_pairwise_matrix()
    test_sample_cov()
    test_pipeline()

if __name__ == "__main__":
    run_all_tests()

test_generate_data_torch
    [PASS] x1 rows sum to 1
    [PASS] x2 rows sum to 1
test_pairwise_matrix_torch
    [PASS] output shape (3, 5, 5)
test_sample_cov_torch
    [PASS] values match numpy reference
test_pipeline_torch
    [PASS] pipeline returned ll = 4219.8027


In [2]:
#autograd_workshop

import numpy as np
import torch
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

def f_numpy(x: np.ndarray) -> np.ndarray:
    return np.sin(x**2) * np.exp(-x) / (1.0 + x**2)

def df_numpy(x: np.ndarray) -> np.ndarray:
    u  = np.sin(x**2) * np.exp(-x)
    du = np.exp(-x) * (2*x * np.cos(x**2) - np.sin(x**2))
    v  = 1.0 + x**2
    dv = 2*x
    return (du * v - u * dv) / v**2

def f_torch(x: torch.Tensor) -> torch.Tensor:
    return torch.sin(x**2) * torch.exp(-x) / (1.0 + x**2)

def df_torch_scalar(x_val: float) -> float:
    x = torch.tensor(x_val, dtype=torch.float64, requires_grad=True)
    y = f_torch(x)
    y.backward()
    return x.grad.item()

def _df_torch_vectorised(xs: np.ndarray) -> np.ndarray:
    x_t = torch.tensor(xs, dtype=torch.float64, requires_grad=True)
    f_torch(x_t).sum().backward()
    return x_t.grad.detach().numpy()

def plot_2_1():
    xs = np.linspace(-2.0, 2.0, 500)
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
    axes[0].plot(xs, f_numpy(xs), lw=2, color="steelblue")
    axes[1].plot(xs, df_numpy(xs), lw=3, label="NumPy hand-coded f′(x)", color="tomato")
    axes[1].plot(xs, _df_torch_vectorised(xs), lw=2, linestyle="--", label="PyTorch autograd f′(x)", color="navy")
    axes[1].legend(fontsize=8)
    plt.tight_layout()
    plt.savefig("2_1_derivative_comparison.png", dpi=130)

def _sigmoid_np(z: np.ndarray) -> np.ndarray:
    return 1.0 / (1.0 + np.exp(-z))

def loss_numpy(w: np.ndarray, X: np.ndarray, y: np.ndarray) -> float:
    y_hat = _sigmoid_np(X @ w)
    return float(np.mean((y_hat - y)**2))

def grad_loss_numpy(w: np.ndarray, X: np.ndarray, y: np.ndarray) -> np.ndarray:
    s = _sigmoid_np(X @ w)
    delta = (s - y) * s * (1.0 - s)
    return (2.0 / len(y)) * (X.T @ delta)

def loss_and_grad_torch(w: torch.Tensor, X: torch.Tensor, y: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    y_hat = torch.sigmoid(X @ w)
    loss = torch.mean((y_hat - y)**2)

    if w.grad is not None:
        w.grad.zero_()
    loss.backward()
    grad = w.grad.clone()

    return loss, grad

def _make_synthetic(seed: int = 7, N: int = 60, D: int = 5):
    rng = np.random.default_rng(seed)
    X   = rng.standard_normal((N, D)).astype(np.float32)
    y   = (rng.uniform(size=N) > 0.5).astype(np.float32)
    w   = (rng.standard_normal(D) * 0.1).astype(np.float32)
    return X, y, w

def test_2_1_derivative_match():
    xs = np.linspace(-2.0, 2.0, 300)
    assert np.allclose(df_numpy(xs), _df_torch_vectorised(xs), atol=1e-6)

def test_2_2_loss_value():
    X, y, w = _make_synthetic()
    X_t, y_t, w_t = torch.tensor(X, dtype=torch.float32), torch.tensor(y, dtype=torch.float32), torch.tensor(w, dtype=torch.float32, requires_grad=True)
    loss_t, _ = loss_and_grad_torch(w_t, X_t, y_t)
    assert abs(loss_t.item() - loss_numpy(w, X, y)) < 1e-5

def test_2_2_gradient_match():
    X, y, w = _make_synthetic()
    X_t, y_t, w_t = torch.tensor(X, dtype=torch.float32), torch.tensor(y, dtype=torch.float32), torch.tensor(w, dtype=torch.float32, requires_grad=True)
    _, grad_t = loss_and_grad_torch(w_t, X_t, y_t)
    assert torch.allclose(grad_t, torch.tensor(grad_loss_numpy(w, X, y), dtype=torch.float32), atol=1e-4)

if __name__ == "__main__":
    test_2_1_derivative_match()
    plot_2_1()
    test_2_2_loss_value()
    test_2_2_gradient_match()

In [4]:
#fcnn

import matplotlib
matplotlib.use("Agg")
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
import matplotlib.pyplot as plt

SEED       = 0
N_EPOCHS   = 500
LR         = 0.05
HIDDEN_DIM = 16

rng = np.random.default_rng(SEED)
torch.manual_seed(SEED)

def load_moons_dataset_from_hardcoded_path():
    # In a Colab environment, __file__ is not defined.
    # We can directly refer to the dataset path if it's uploaded to /content/.
    dataset_path = Path("/content/moons_dataset.npz")
    if not dataset_path.exists():
        raise FileNotFoundError("Run aux/data/load_data.py first or ensure moons_dataset.npz is in /content/.")
    npz_data = np.load(dataset_path)
    return npz_data["x_train"], npz_data["y_train"]

try:
    X_np, y_np = load_moons_dataset_from_hardcoded_path()
    X_np = X_np.astype(np.float32)
    y_np = y_np.astype(np.float32)
    X_t = torch.tensor(X_np)
    y_t = torch.tensor(y_np)
except FileNotFoundError:
    X_np, y_np = np.zeros((10,2), dtype=np.float32), np.zeros(10, dtype=np.float32)
    X_t, y_t = torch.tensor(X_np), torch.tensor(y_np)

class FeedForwardNumPy:
    def __init__(self, input_dim: int, hidden_dim: int, output_dim: int, seed: int = 0):
        _rng   = np.random.default_rng(seed)
        scale  = np.sqrt(2.0 / input_dim)
        self.W1 = _rng.normal(0, scale, (hidden_dim, input_dim)).astype(np.float32)
        self.b1 = np.zeros(hidden_dim, dtype=np.float32)
        scale2  = np.sqrt(2.0 / hidden_dim)
        self.W2 = _rng.normal(0, scale2, (hidden_dim, hidden_dim)).astype(np.float32)
        self.b2 = np.zeros(hidden_dim, dtype=np.float32)
        self.W3 = _rng.normal(0, scale2, (output_dim, hidden_dim)).astype(np.float32)
        self.b3 = np.zeros(output_dim, dtype=np.float32)

    @staticmethod
    def relu(x: np.ndarray) -> np.ndarray:
        return np.maximum(0.0, x)

    def forward(self, x: np.ndarray) -> np.ndarray:
        h1  = self.relu(x  @ self.W1.T + self.b1)
        h2  = self.relu(h1 @ self.W2.T + self.b2)
        out = h2 @ self.W3.T + self.b3
        return out

class FeedForwardNN(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int, output_dim: int):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, output_dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        out = self.fc3(x)
        return out

def train(model: nn.Module, X: torch.Tensor, y: torch.Tensor, lr: float = 0.05, n_epochs: int = 500) -> list:
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)
    loss_fn   = nn.BCEWithLogitsLoss()
    losses    = []
    for _ in range(n_epochs):
        logits = model(X).squeeze(1)
        loss   = loss_fn(logits, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    return losses

if __name__ == "__main__":
    torch.manual_seed(SEED)
    torch_model = FeedForwardNN(2, HIDDEN_DIM, 1)
    if X_t.shape[0] > 10:
        losses = train(torch_model, X_t, y_t, lr=LR, n_epochs=N_EPOCHS)
        print("Model Trained Successfully.")

Model Trained Successfully.


In [5]:
#classifiers

import numpy as np
import torch
import torch.nn as nn
from torch import optim

torch.manual_seed(42)
rng = np.random.default_rng(42)

def _generate_exoplanet_data(n_samples: int = 1000):
    n_pos = n_samples // 2
    n_neg = n_samples - n_pos

    depth_pos    = rng.beta(2, 5,   n_pos) * 0.08 + 0.002
    duration_pos = rng.normal(4.0, 1.5,  n_pos).clip(1, 20)
    period_pos   = rng.lognormal(3.0, 1.2, n_pos).clip(1, 500)
    radius_pos   = rng.normal(1.0, 0.3,  n_pos).clip(0.1, 3)
    temp_pos     = rng.normal(1200, 400, n_pos).clip(100, 2500)
    snr_pos      = rng.normal(45, 15,   n_pos).clip(5, 100)

    depth_neg    = rng.beta(5, 2,   n_neg) * 0.3 + 0.01
    duration_neg = rng.normal(9.0, 3.5,  n_neg).clip(1, 20)
    period_neg   = rng.lognormal(2.0, 1.8, n_neg).clip(1, 500)
    radius_neg   = rng.normal(1.6, 0.6,  n_neg).clip(0.1, 3)
    temp_neg     = rng.normal(700,  350, n_neg).clip(100, 2500)
    snr_neg      = rng.normal(22, 12,   n_neg).clip(5, 100)

    X_pos = np.stack([depth_pos, duration_pos, period_pos, radius_pos, temp_pos, snr_pos], axis=1)
    X_neg = np.stack([depth_neg, duration_neg, period_neg, radius_neg, temp_neg, snr_neg], axis=1)

    X = np.vstack([X_pos, X_neg]).astype(np.float32)
    y = np.concatenate([np.ones(n_pos), np.zeros(n_neg)]).astype(np.float32)

    shuffle = rng.permutation(n_samples)
    return X[shuffle], y[shuffle]

X_all, y_all = _generate_exoplanet_data(n_samples=1000)

split      = int(0.8 * len(X_all))
X_train_np = X_all[:split]
y_train_np = y_all[:split]
X_val_np   = X_all[split:]
y_val_np   = y_all[split:]

def make_batches(X: torch.Tensor, y: torch.Tensor, batch_size: int):
    n   = X.shape[0]
    idx = torch.randperm(n)
    for start in range(0, n, batch_size):
        batch_idx = idx[start : start + batch_size]
        yield X[batch_idx], y[batch_idx]

class ExoplanetClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(6, 32)
        self.fc2 = nn.Linear(32, 16)
        self.fc3 = nn.Linear(16, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        logits = self.fc3(x).squeeze(1)
        return logits

def standardize(X_train: np.ndarray, X_val: np.ndarray) -> tuple:
    mean = X_train.mean(axis=0)
    std = X_train.std(axis=0)
    X_train_std = (X_train - mean) / (std + 1e-8)
    X_val_std = (X_val - mean) / (std + 1e-8)
    return X_train_std, X_val_std

def compute_accuracy(logits: torch.Tensor, labels: torch.Tensor) -> float:
    probs = torch.sigmoid(logits)
    predictions = probs >= 0.5
    accuracy = (predictions == labels.bool()).float().mean().item()
    return accuracy

def train_classifier(
    model,
    X_train:    np.ndarray,
    y_train:    np.ndarray,
    X_val:      np.ndarray,
    y_val:      np.ndarray,
    lr:         float = 1e-3,
    batch_size: int   = 32,
    num_epochs: int   = 150,
) -> tuple:

    X_train_std, X_val_std = standardize(X_train, X_val)

    X_tr = torch.tensor(X_train_std, dtype=torch.float32)
    y_tr = torch.tensor(y_train, dtype=torch.float32)
    X_v = torch.tensor(X_val_std, dtype=torch.float32)
    y_v = torch.tensor(y_val, dtype=torch.float32)

    loss_fn = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    train_losses = []
    val_accuracies = []

    for epoch in range(num_epochs):
        batch_losses = []
        for X_batch, y_batch in make_batches(X_tr, y_tr, batch_size):
            logits = model(X_batch)
            loss = loss_fn(logits, y_batch)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            batch_losses.append(loss.item())

        train_losses.append(np.mean(batch_losses))

        with torch.no_grad():
            val_logits = model(X_v)
            val_acc = compute_accuracy(val_logits, y_v)
            val_accuracies.append(val_acc)

    return model, train_losses, val_accuracies

if __name__ == "__main__":
    torch.manual_seed(42)
    model = ExoplanetClassifier()
    model, train_losses, val_accuracies = train_classifier(
        model, X_train_np, y_train_np, X_val_np, y_val_np,
        lr=1e-3, batch_size=32, num_epochs=150,
    )
    print(f"Final Validation Accuracy: {val_accuracies[-1]:.3f}")

Final Validation Accuracy: 1.000
